# 05 — Experiment & Uplift Lab
Treatment/control, uplift, incremental outcomes, costo incremental y gates experimentales.


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
import pandas as pd
import numpy as np
cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")
SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))
from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
settings=load_settings(PROJECT_ROOT)
conn=connect_postgres(settings)
def sql_df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)
print("DB:",settings.postgres.database)


## 1. Dataset experimental


In [ ]:
exp=sql_df("""
SELECT
 r.recommendation_id,
 r.entity_id AS lead_id,
 r.created_at AS recommendation_at,
 a.action_taken,
 a.action_cost,
 a.action_at,
 o.outcome_name,
 o.outcome_value
FROM decision_intelligence.recommendations r
LEFT JOIN decision_intelligence.actions a
  ON a.recommendation_id=r.recommendation_id
LEFT JOIN decision_intelligence.outcomes o
  ON o.entity_id=r.entity_id
WHERE r.decision_system='priorizacion_leads'
""")
print("rows:",len(exp))
display(exp.head())


## 2. Treatment/control


In [ ]:
if len(exp):
    exp["treated"]=exp["action_taken"].notna().astype(int)
    display(exp["treated"].value_counts(dropna=False))


## 3. Uplift simple


In [ ]:
if len(exp):
    e=exp.dropna(subset=["outcome_value"]).copy()
    if len(e) and e["treated"].nunique()==2:
        rates=e.groupby("treated")["outcome_value"].mean()
        uplift=rates.loc[1]-rates.loc[0]
        print("Treatment:",rates.loc[1])
        print("Control:",rates.loc[0])
        print("Uplift:",uplift)


## 4. Incremental outcomes / costo


In [ ]:
if "uplift" in globals():
    treated_n=int((e["treated"]==1).sum())
    incremental_outcomes=treated_n*uplift
    total_cost=e.loc[e["treated"]==1,"action_cost"].fillna(0).sum()
    print("Incremental outcomes estimados:",incremental_outcomes)
    print("Costo total:",total_cost)
    if incremental_outcomes>0:
        print("Costo por outcome incremental:",total_cost/incremental_outcomes)


## 5. Gate experimental


In [ ]:
gates=[
{"gate":"Treatment/control disponibles","status":"PASS" if len(exp) and exp.get("treated",pd.Series()).nunique()==2 else "PENDING"},
{"gate":"Outcome definido","status":"PASS" if len(exp) and exp["outcome_value"].notna().sum()>20 else "PENDING"},
{"gate":"Asignación aleatoria o diseño defendible","status":"PENDING"},
{"gate":"Sample size suficiente","status":"PENDING"}
]
pd.DataFrame(gates)


In [ ]:
conn.close(); print("Conexión cerrada.")
